# Single-time Fourier transform of the center-site correlator

This file has one fixed `t_prime`, so this computes the **one-sided** transform $F(\omega)=\int_{t'}^{t_{\max}}dt\,G(t',t)e^{-i\omega(t-t')}$ for `center_site == site_index`. An optional Gaussian window suppresses the finite-time cutoff. This is not the two-time probe integral, and $G^<$ alone is not a retarded Green function. The output is the complex transform, not automatically a spectral function. Change the exponent sign if required by your convention.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CSV_FILE = Path('/Users/qqt/Library/CloudStorage/OneDrive-OakRidgeNationalLaboratory/Research_Projects/r_stcvsttc/code/template_folder_hubbard_chain_cv/N=8_Nup=4_Ndown=4_20261023_140645/G<_singletime_N=8_Nup=4_Ndown=4_20261023_140645_steps=0_P2_c_P3_centersite=3_time=0.0.csv')
CENTER_SITE = 3
omega_axis = np.arange(-20, 20.01, 0.1)  # angular frequency
EXPONENT_SIGN = -1  # kernel exp(-i omega (t-t_prime)); use +1 if appropriate
GAUSSIAN_SIGMA = None  # e.g. 10.0 for a cutoff-suppressing Gaussian; None for no window


In [ ]:
def load_center_site_series(csv_file, center_site):
    """Load G(t_prime, t) for one fixed t_prime and center_site == site_index."""
    data = pd.read_csv(csv_file)
    required_columns = {
        "t_prime", "t", "center_site", "site_index",
        "real_part", "imaginary_part",
    }
    missing_columns = required_columns - set(data.columns)
    if missing_columns:
        raise ValueError(f"CSV is missing columns: {sorted(missing_columns)}")

    # Keep only the center-site-to-center-site correlator.
    selected = data.loc[
        (data["center_site"] == center_site)
        & (data["site_index"] == center_site)
    ].copy()
    if selected.empty:
        raise ValueError(f"No data found for center site {center_site}.")

    # This transform handles one fixed t_prime at a time.
    if selected["t_prime"].nunique() != 1:
        raise ValueError("Expected exactly one t_prime value in the selected data.")
    if selected["t"].duplicated().any():
        raise ValueError("Duplicate time samples found for the selected site.")

    selected = selected.sort_values("t")
    t_prime = float(selected["t_prime"].iloc[0])
    tau = selected["t"].to_numpy(dtype=float) - t_prime
    correlator = (
        selected["real_part"].to_numpy(dtype=float)
        + 1j * selected["imaginary_part"].to_numpy(dtype=float)
    )

    if len(tau) < 2:
        raise ValueError("At least two time samples are required.")
    if tau[0] < -1e-9 or not np.all(np.diff(tau) > 0):
        raise ValueError("Times must be increasing and satisfy t >= t_prime.")
    if not np.all(np.isfinite(correlator)):
        raise ValueError("Correlator contains non-finite values.")

    return t_prime, tau, correlator


def fourier_transform_time_series(
    tau,
    correlator,
    omega_axis,
    *,
    exponent_sign=-1,
    gaussian_sigma=None,
):
    """Compute F(omega) = integral d tau G(tau) exp(sign*i*omega*tau)."""
    tau = np.asarray(tau, dtype=float)
    correlator = np.asarray(correlator, dtype=complex)
    omega_axis = np.asarray(omega_axis, dtype=float)

    if exponent_sign not in (-1, 1):
        raise ValueError("exponent_sign must be -1 or +1.")
    if tau.ndim != 1 or correlator.shape != tau.shape or len(tau) < 2:
        raise ValueError("tau and correlator must be matching 1D arrays with at least 2 samples.")
    if not np.all(np.diff(tau) > 0):
        raise ValueError("tau must be strictly increasing.")
    if not np.all(np.isfinite(omega_axis)):
        raise ValueError("omega_axis must contain only finite values.")
    if gaussian_sigma is not None and (
        not np.isfinite(gaussian_sigma) or gaussian_sigma <= 0
    ):
        raise ValueError("gaussian_sigma must be positive and finite.")

    # Optional window reduces artifacts from the finite time interval.
    if gaussian_sigma is None:
        window = np.ones_like(tau)
    else:
        window = np.exp(-0.5 * (tau / gaussian_sigma) ** 2)

    # Integrate each frequency using the actual (possibly nonuniform) time grid.
    transform = np.empty(len(omega_axis), dtype=complex)
    for index, omega in enumerate(omega_axis):
        integrand = correlator * window * np.exp(
            exponent_sign * 1j * omega * tau
        )
        transform[index] = np.trapezoid(integrand, x=tau)

    return transform


# Load the selected time trace, then Fourier transform it.
t_prime, tau, correlator = load_center_site_series(CSV_FILE, CENTER_SITE)
F_omega = fourier_transform_time_series(
    tau,
    correlator,
    omega_axis,
    exponent_sign=EXPONENT_SIGN,
    gaussian_sigma=GAUSSIAN_SIGMA,
)

print(
    f"Loaded {len(tau)} samples for site {CENTER_SITE}; "
    f"t_prime={t_prime:g}, tau range=[{tau[0]:g}, {tau[-1]:g}]"
)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), constrained_layout=True)
axes[0].plot(tau, correlator.real, label='Re G')
axes[0].plot(tau, correlator.imag, label='Im G')
axes[0].set(xlabel="t - t'", ylabel='G', title='Center-site time series')
axes[0].legend()
axes[1].plot(omega_axis, F_omega.real, label='Re F')
axes[1].plot(omega_axis, F_omega.imag, label='Im F')
axes[1].set(xlabel='Angular frequency ω', ylabel='F(ω)', title='One-sided Fourier transform')
axes[1].legend()
plt.show()


In [ ]:
output_csv = CSV_FILE.with_name(CSV_FILE.stem + '_fourier.csv')
pd.DataFrame({'omega': omega_axis, 't_prime': t_prime, 'center_site': CENTER_SITE,
              'fourier_real': F_omega.real, 'fourier_imag': F_omega.imag}).to_csv(output_csv, index=False)
print('Saved:', output_csv)
